# Cuando el camino más corto no es el mejor — Modelado del terreno y función de costo

**Problema:** evacuar una zona inundada hacia un refugio. La ruta más corta puede cruzar zonas de alto riesgo.

**Pregunta eje:** *Si dos rutas llegan al mismo refugio, pero una es más corta y la otra atraviesa zonas de mayor riesgo, ¿cómo determina el sistema cuál es la mejor?*
Comparando el costo acumulado $J(P)=\sum C(u,v)$, no la longitud.

## Función de costo
$$C(u,v)=\begin{cases} d(u,v)\,\bigl(1+\alpha\,R(v)^{\beta}\bigr) & v \text{ transitable}\\ \infty & R(v)=-1\end{cases}$$

- $d(u,v)$: 1 (ortogonal) o $\sqrt2$ (diagonal).
- $R(v)$: nivel de riesgo de la celda destino (0 seco, 1–3 anegación, -1 bloqueo total).
- $\alpha$: aversión al riesgo. $\beta$: exponente no lineal.

| Nivel | Significado | Factor (α=2, β=1.5) |
|---|---|---|
| 0 | Seco | 1.00 |
| 1 | Leve | 3.00 |
| 2 | Moderado | 6.66 |
| 3 | Severo | 11.39 |
| -1 | Bloqueo | ∞ |


In [ ]:
from __future__ import annotations

import math

import numpy as np

Node = tuple[int, int]


class GridEnvironment:
    """Cuadrícula de riesgo de inundación con función de costo de transición.

    Niveles: 0 seco, 1-3 anegación creciente, -1 bloqueo total.
    Costo: C(u, v) = d(u, v) * (1 + alpha * R(v) ** beta); inf si v está bloqueada.
    """

    BLOCKED = -1
    MAX_RISK = 3
    _ORTHO = ((-1, 0), (1, 0), (0, -1), (0, 1))
    _DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))

    def __init__(self, rows: int, cols: int, alpha: float = 2.0,
                 beta: float = 1.5, allow_diagonal: bool = False) -> None:
        if rows <= 0 or cols <= 0:
            raise ValueError("rows y cols deben ser positivos")
        if alpha < 0 or beta < 1:
            raise ValueError("se requiere alpha >= 0 y beta >= 1")
        self.rows = rows
        self.cols = cols
        self.alpha = alpha
        self.beta = beta
        self.allow_diagonal = allow_diagonal
        self.grid: np.ndarray = np.zeros((rows, cols), dtype=np.int8)

    # ---- validación ----
    def in_bounds(self, r: int, c: int) -> bool:
        """True si (r, c) está dentro de la cuadrícula."""
        return 0 <= r < self.rows and 0 <= c < self.cols

    def _check_level(self, risk_level: int) -> None:
        if risk_level != self.BLOCKED and not 0 <= risk_level <= self.MAX_RISK:
            raise ValueError(f"risk_level inválido: {risk_level} (use -1 o 0..3)")

    # ---- modificación del terreno ----
    def set_cell(self, r: int, c: int, risk_level: int) -> None:
        """Asigna el nivel de riesgo de una celda."""
        if not self.in_bounds(r, c):
            raise IndexError(f"celda fuera de límites: ({r}, {c})")
        self._check_level(risk_level)
        self.grid[r, c] = risk_level

    def set_region_risk(self, r_start: int, r_end: int, c_start: int,
                        c_end: int, risk_level: int) -> None:
        """Asigna un nivel a la región rectangular [r_start, r_end] x [c_start, c_end] (inclusive)."""
        if r_start > r_end or c_start > c_end:
            raise ValueError("rango de región invertido")
        if not (self.in_bounds(r_start, c_start) and self.in_bounds(r_end, c_end)):
            raise IndexError("región fuera de límites")
        self._check_level(risk_level)
        self.grid[r_start:r_end + 1, c_start:c_end + 1] = risk_level

    def load_matrix(self, matrix: np.ndarray | list[list[int]]) -> None:
        """Carga una matriz 2D completa validando dimensiones y niveles (-1 o 0..3)."""
        arr = np.asarray(matrix, dtype=np.int8)
        if arr.shape != (self.rows, self.cols):
            raise ValueError(f"dimensiones inválidas: se esperaba {(self.rows, self.cols)}, se recibió {arr.shape}")
        if not np.isin(arr, (-1, 0, 1, 2, 3)).all():
            raise ValueError("la matriz contiene niveles fuera de {-1, 0, 1, 2, 3}")
        self.grid[:, :] = arr

    def reset(self) -> None:
        """Limpia el terreno a ceros (todo seco)."""
        self.grid.fill(0)

    # ---- consultas ----
    def is_blocked(self, r: int, c: int) -> bool:
        """True si la celda es intransitable (bloqueada o fuera de límites)."""
        return (not self.in_bounds(r, c)) or self.grid[r, c] == self.BLOCKED

    def cost(self, u: Node, v: Node) -> float:
        """Costo de transición u -> v (v adyacente). inf si v está bloqueada."""
        if self.is_blocked(*v):
            return math.inf
        dr, dc = abs(u[0] - v[0]), abs(u[1] - v[1])
        if max(dr, dc) != 1:
            raise ValueError(f"{u} y {v} no son adyacentes")
        d = math.sqrt(2.0) if dr + dc == 2 else 1.0
        risk = float(self.grid[v])
        return d * (1.0 + self.alpha * risk ** self.beta)

    def get_neighbors(self, node: Node) -> list[tuple[Node, float]]:
        """Vecinos transitables de `node` como ((r, c), costo). Nunca incluye bloqueados."""
        r, c = node
        if not self.in_bounds(r, c):
            raise IndexError(f"nodo fuera de límites: {node}")
        result: list[tuple[Node, float]] = []
        for dr, dc in self._ORTHO:
            nr, nc = r + dr, c + dc
            if not self.is_blocked(nr, nc):
                result.append(((nr, nc), self.cost(node, (nr, nc))))
        if self.allow_diagonal:
            for dr, dc in self._DIAG:
                nr, nc = r + dr, c + dc
                if self.is_blocked(nr, nc):
                    continue
                # Anti corner-cutting: impide diagonal cuando las dos ortogonales intermedias están bloqueadas
                if self.is_blocked(r + dr, c) and self.is_blocked(r, c + dc):
                    continue
                result.append(((nr, nc), self.cost(node, (nr, nc))))
        return result


In [ ]:
# Smoke test: mapa 5x5 con un bloqueo
env = GridEnvironment(5, 5)
env.set_cell(2, 2, -1)
assert env.is_blocked(2, 2)
nb = dict(env.get_neighbors((2, 1)))
assert (2, 2) not in nb and set(nb) == {(1, 1), (3, 1), (2, 0)}, nb
print("Smoke test OK:", nb)

# Costos ponderados (alpha=2, beta=1.5)
env.reset()
env.set_cell(1, 0, 1)
env.set_cell(0, 1, 3)
assert math.isclose(env.cost((0, 0), (1, 0)), 1 + 2 * 1 ** 1.5)      # 3.0
assert math.isclose(env.cost((0, 0), (0, 1)), 1 + 2 * 3 ** 1.5)      # ~11.39
assert env.cost((0, 0), (1, 0)) < env.cost((0, 0), (0, 1))           # más riesgo, más costo
env.set_cell(0, 1, -1)
assert env.cost((0, 0), (0, 1)) == math.inf
assert (0, 1) not in dict(env.get_neighbors((0, 0)))

# Región, carga matricial y reset
env.set_region_risk(0, 1, 0, 1, 2)
assert (env.grid[0:2, 0:2] == 2).all()
env.load_matrix(np.zeros((5, 5), dtype=np.int8))
assert (env.grid == 0).all()
env.reset()
assert (env.grid == 0).all()

# Diagonales: costo sqrt(2) y anti corner-cutting
d = GridEnvironment(3, 3, allow_diagonal=True)
assert math.isclose(dict(d.get_neighbors((1, 1)))[(0, 0)], math.sqrt(2))
d.set_cell(0, 1, -1)
d.set_cell(1, 0, -1)
assert (0, 0) not in dict(d.get_neighbors((1, 1)))   # ambas bloqueadas -> impide corte de esquina
d.set_cell(1, 0, 0)
assert (0, 0) in dict(d.get_neighbors((1, 1)))       # solo una bloqueada -> permitido

# Validaciones
for bad in (lambda: env.set_cell(9, 9, 0), lambda: env.set_cell(0, 0, 7)):
    try:
        bad()
        raise AssertionError("debió fallar")
    except (IndexError, ValueError):
        pass
print("Todas las pruebas pasaron")


## Módulo 2: Algoritmos de búsqueda (Dijkstra y A*) y conteo de nodos explorados

### 1. Comparación técnica: Dijkstra vs A*
Ambos algoritmos resuelven el problema de encontrar una ruta óptima desde el origen `start` hasta el refugio `goal` minimizando el **costo acumulado** $J(P) = \sum_{(u,v) \in P} C(u,v)$ sobre el grafo implícito generado por `GridEnvironment.get_neighbors(u)`:

- **Algoritmo de Dijkstra (búsqueda de costo uniforme):** Ordena la frontera de exploración (cola de prioridad) únicamente por el costo acumulado desde el inicio, $f(n) = g(n)$. Dado que todas las celdas transitables tienen costo estrictamente positivo $C(u,v) \ge d(u,v) \ge 1 > 0$, la primera vez que un nodo se extrae de la cola de prioridad su costo $g(n)$ es óptimo. Sin embargo, al carecer de información direccional hacia la meta, expande contornos de igual costo en todas las direcciones.
- **Algoritmo A* (búsqueda informada):** Incorpora una heurística $h(n)$ que estima el costo mínimo restante desde $n$ hasta `goal`, priorizando los nodos mediante $f(n) = g(n) + h(n)$. Esto focaliza la búsqueda hacia el refugio y reduce los nodos expandidos (`nodes_explored`) garantizando el mismo costo óptimo que Dijkstra siempre que $h(n)$ sea admisible.

### 2. Definición formal de la heurística admisible $h(n)$
Sea $n = (r_n, c_n)$ el nodo actual y $g = (r_g, c_g)$ el refugio (`goal`), con $\Delta r = |r_n - r_g|$ y $\Delta c = |c_n - c_g|$. La heurística se adapta a la topología del entorno:

1. **Sin diagonales (`allow_diagonal = False`) — Distancia Manhattan:**
   $$h_{\text{Manhattan}}(n, g) = \Delta r + \Delta c$$
2. **Con diagonales (`allow_diagonal = True`) — Distancia Octile / Euclidiana:**
   $$h_{\text{Octile}}(n, g) = \max(\Delta r, \Delta c) + (\sqrt{2} - 1)\,\min(\Delta r, \Delta c)$$
   *(Nota: la distancia Euclidiana $h_{\text{Euclidiana}}(n, g) = \sqrt{\Delta r^2 + \Delta c^2}$ también es admisible al cumplir $h_{\text{Euclidiana}}(n, g) \le h_{\text{Octile}}(n, g)$; la distancia Octile representa la distancia mínima exacta en una cuadrícula 8-conexa con pasos ortogonales de costo $1$ y diagonales de costo $\sqrt{2}$).*

### 3. Justificación matemática de admisibilidad ($h(n) \le h^*(n)$)
Para toda celda transitable $v$, el riesgo cumple $R(v) \in \{0, 1, 2, 3\}$, con $\alpha \ge 0$ y $\beta \ge 1$. Por consiguiente, el factor de penalización por inundación verifica siempre:
$$1 + \alpha\,R(v)^{\beta} \ge 1$$
Al multiplicar por la distancia base $d(u,v) \in \{1, \sqrt{2}\}$, cada transición transitable satisface:
$$C(u, v) = d(u, v)\,\bigl(1 + \alpha\,R(v)^{\beta}\bigr) \ge d(u, v)$$
Sumando sobre cualquier camino válido $P$ de $n$ a $g$, el costo real acumulado cumple:
$$J(P) = \sum_{(u,v) \in P} C(u,v) \ge \sum_{(u,v) \in P} d(u,v) \ge h(n, g)$$
Tomando el mínimo sobre todos los caminos posibles se obtiene $h(n, g) \le h^*(n, g)$ (**admisibilidad**) y $h(u, g) \le d(u,v) + h(v, g) \le C(u,v) + h(v, g)$ (**consistencia / monotonía**). Por lo tanto, $h(n)$ **nunca sobreestima** el costo real al refugio y A* encuentra garantizadamente la ruta óptima.


In [ ]:
from dataclasses import dataclass
import heapq


@dataclass
class SearchResult:
    """Resultado estándar de búsqueda para consumo de pruebas y visualización.

    Permite acceso tanto por atributos (`res.cost`) como por llave de diccionario
    (`res["cost"]`) para facilitar la integración con los demás módulos.
    """

    path: list[Node]
    cost: float
    nodes_explored: int
    visited_order: list[Node]

    def __getitem__(self, key: str) -> list[Node] | float | int:
        if key in ("path", "cost", "nodes_explored", "visited_order"):
            return getattr(self, key)
        raise KeyError(key)

    def __contains__(self, key: object) -> bool:
        return key in ("path", "cost", "nodes_explored", "visited_order")

    def keys(self) -> tuple[str, str, str, str]:
        return ("path", "cost", "nodes_explored", "visited_order")

    def get(self, key: str, default: object = None) -> object:
        return getattr(self, key, default) if key in self else default

    def to_dict(self) -> dict[str, list[Node] | float | int]:
        return {
            "path": self.path,
            "cost": self.cost,
            "nodes_explored": self.nodes_explored,
            "visited_order": self.visited_order,
        }


def heuristic(node: Node, goal: Node, allow_diagonal: bool = False) -> float:
    """Heurística admisible y consistente h(n) según la conectividad del entorno.

    - allow_diagonal=False: distancia Manhattan.
    - allow_diagonal=True: distancia Octile (cota inferior en malla 8-conexa).
    """
    dr = abs(node[0] - goal[0])
    dc = abs(node[1] - goal[1])
    if not allow_diagonal:
        return float(dr + dc)
    return max(dr, dc) + (math.sqrt(2.0) - 1.0) * min(dr, dc)


def _reconstruct_path(came_from: dict[Node, Node], start: Node, goal: Node) -> list[Node]:
    """Reconstruye la ruta ordenada desde `start` hasta `goal`."""
    path: list[Node] = [goal]
    current = goal
    while current != start:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path


def dijkstra(env: GridEnvironment, start: Node, goal: Node) -> SearchResult:
    """Algoritmo de Dijkstra (costo uniforme) usando la interfaz de GridEnvironment."""
    if env.is_blocked(*start) or env.is_blocked(*goal):
        return SearchResult(path=[], cost=math.inf, nodes_explored=0, visited_order=[])

    open_heap: list[tuple[float, int, Node]] = [(0.0, 0, start)]
    came_from: dict[Node, Node] = {}
    g_score: dict[Node, float] = {start: 0.0}
    closed: set[Node] = set()
    visited_order: list[Node] = []
    nodes_explored = 0
    tie_breaker = 0

    while open_heap:
        current_cost, _, current = heapq.heappop(open_heap)
        if current in closed:
            continue
        closed.add(current)
        visited_order.append(current)
        nodes_explored += 1

        if current == goal:
            return SearchResult(
                path=_reconstruct_path(came_from, start, goal),
                cost=current_cost,
                nodes_explored=nodes_explored,
                visited_order=visited_order,
            )

        for neighbor, step_cost in env.get_neighbors(current):
            if neighbor in closed:
                continue
            tentative_g = current_cost + step_cost
            if tentative_g < g_score.get(neighbor, math.inf):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                tie_breaker += 1
                heapq.heappush(open_heap, (tentative_g, tie_breaker, neighbor))

    return SearchResult(
        path=[], cost=math.inf, nodes_explored=nodes_explored, visited_order=visited_order
    )


def a_star(env: GridEnvironment, start: Node, goal: Node) -> SearchResult:
    """Algoritmo A* con heurística admisible adaptada a `env.allow_diagonal`."""
    if env.is_blocked(*start) or env.is_blocked(*goal):
        return SearchResult(path=[], cost=math.inf, nodes_explored=0, visited_order=[])

    h_start = heuristic(start, goal, env.allow_diagonal)
    open_heap: list[tuple[float, float, int, Node]] = [(h_start, h_start, 0, start)]
    came_from: dict[Node, Node] = {}
    g_score: dict[Node, float] = {start: 0.0}
    closed: set[Node] = set()
    visited_order: list[Node] = []
    nodes_explored = 0
    tie_breaker = 0

    while open_heap:
        _, _, _, current = heapq.heappop(open_heap)
        if current in closed:
            continue
        closed.add(current)
        visited_order.append(current)
        nodes_explored += 1

        if current == goal:
            return SearchResult(
                path=_reconstruct_path(came_from, start, goal),
                cost=g_score[current],
                nodes_explored=nodes_explored,
                visited_order=visited_order,
            )

        for neighbor, step_cost in env.get_neighbors(current):
            if neighbor in closed:
                continue
            tentative_g = g_score[current] + step_cost
            if tentative_g < g_score.get(neighbor, math.inf):
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                h_val = heuristic(neighbor, goal, env.allow_diagonal)
                f_val = tentative_g + h_val
                tie_breaker += 1
                heapq.heappush(open_heap, (f_val, h_val, tie_breaker, neighbor))

    return SearchResult(
        path=[], cost=math.inf, nodes_explored=nodes_explored, visited_order=visited_order
    )


In [ ]:
# Smoke Test de Búsqueda (Módulo 2: Dijkstra y A*)

# 1. Ruta recta con riesgo alto (nivel 3) vs rodeo por celdas secas (nivel 0)
env_rodeo = GridEnvironment(5, 5, alpha=2.0, beta=1.5, allow_diagonal=False)
# Línea directa en fila 0 ((0,1), (0,2), (0,3)) con anegación severa (nivel 3)
env_rodeo.set_region_risk(0, 0, 1, 3, 3)
start_node, goal_node = (0, 0), (0, 4)

res_dijkstra_rodeo = dijkstra(env_rodeo, start_node, goal_node)
res_astar_rodeo = a_star(env_rodeo, start_node, goal_node)

# Costo directo por fila 0 sería: 3 * (1 + 2 * 3^1.5) + 1 ≈ 35.18 (4 pasos)
# Rodeo seco por fila 1 toma 6 pasos de costo 1.0 -> costo total = 6.0
direct_risky_cost = sum(env_rodeo.cost((0, c), (0, c + 1)) for c in range(4))
assert math.isclose(res_dijkstra_rodeo["cost"], 6.0), res_dijkstra_rodeo
assert math.isclose(res_astar_rodeo["cost"], 6.0), res_astar_rodeo
assert res_dijkstra_rodeo["cost"] < direct_risky_cost
assert res_dijkstra_rodeo["path"] == res_astar_rodeo["path"]
assert all(env_rodeo.grid[r, c] == 0 for r, c in res_astar_rodeo["path"])
print(
    f"Prueba 1 OK (Rodeo seguro): costo óptimo = {res_astar_rodeo['cost']:.2f} "
    f"vs ruta recta riesgosa = {direct_risky_cost:.2f} | pasos = {len(res_astar_rodeo['path']) - 1}"
)

# 2. Comparación de nodes_explored entre Dijkstra y A* (ortogonal y diagonal)
env_comp = GridEnvironment(10, 10, alpha=2.0, beta=1.5, allow_diagonal=False)
env_comp.set_region_risk(2, 6, 3, 5, 2)  # zona de riesgo moderado
env_comp.set_region_risk(4, 4, 1, 6, 3)  # franja de riesgo severo

res_d = dijkstra(env_comp, (0, 0), (8, 8))
res_a = a_star(env_comp, (0, 0), (8, 8))
assert math.isclose(res_d["cost"], res_a["cost"])
assert res_a["nodes_explored"] <= res_d["nodes_explored"]
assert len(res_a["visited_order"]) == res_a["nodes_explored"]
print(
    f"Prueba 2a OK (Ortogonal): costo Dijkstra = {res_d['cost']:.2f}, A* = {res_a['cost']:.2f} | "
    f"explorados Dijkstra = {res_d['nodes_explored']}, A* = {res_a['nodes_explored']}"
)

env_diag = GridEnvironment(10, 10, alpha=2.0, beta=1.5, allow_diagonal=True)
env_diag.set_region_risk(2, 6, 3, 5, 2)
res_d_diag = dijkstra(env_diag, (0, 0), (8, 8))
res_a_diag = a_star(env_diag, (0, 0), (8, 8))
assert math.isclose(res_d_diag["cost"], res_a_diag["cost"])
assert res_a_diag["nodes_explored"] <= res_d_diag["nodes_explored"]
print(
    f"Prueba 2b OK (Diagonal):  costo Dijkstra = {res_d_diag['cost']:.2f}, A* = {res_a_diag['cost']:.2f} | "
    f"explorados Dijkstra = {res_d_diag['nodes_explored']}, A* = {res_a_diag['nodes_explored']}"
)

# 3. Destino encerrado por bloqueos (-1) y nodos inicio/meta bloqueados
env_blocked = GridEnvironment(5, 5, allow_diagonal=True)
# Encerramos el refugio (4, 4) con celdas bloqueadas (-1)
for r_b, c_b in ((3, 4), (4, 3), (3, 3)):
    env_blocked.set_cell(r_b, c_b, -1)

res_d_enc = dijkstra(env_blocked, (0, 0), (4, 4))
res_a_enc = a_star(env_blocked, (0, 0), (4, 4))
assert res_d_enc["path"] == [] and res_d_enc["cost"] == math.inf
assert res_a_enc["path"] == [] and res_a_enc["cost"] == math.inf

# Inicio o destino directamente bloqueado
env_blocked.set_cell(0, 0, -1)
assert dijkstra(env_blocked, (0, 0), (2, 2)).to_dict() == {
    "path": [], "cost": math.inf, "nodes_explored": 0, "visited_order": []
}
assert a_star(env_blocked, (0, 0), (2, 2)).to_dict() == {
    "path": [], "cost": math.inf, "nodes_explored": 0, "visited_order": []
}
print("Prueba 3 OK (Bloqueos absolutos): manejo limpio sin ciclos (path=[], cost=inf)")
print("Todas las pruebas del Módulo 2 pasaron exitosamente.")


Prueba 1 OK (Rodeo seguro): costo óptimo = 6.00 vs ruta recta riesgosa = 35.18 | pasos = 6
Prueba 2a OK (Ortogonal): costo Dijkstra = 16.00, A* = 16.00 | explorados Dijkstra = 86, A* = 17
Prueba 2b OK (Diagonal):  costo Dijkstra = 13.66, A* = 13.66 | explorados Dijkstra = 89, A* = 38
Prueba 3 OK (Bloqueos absolutos): manejo limpio sin ciclos (path=[], cost=inf)
Todas las pruebas del Módulo 2 pasaron exitosamente.
